# Annotator Agreement

Cohen's kappa and observed agreement for the four annotated features.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import cohen_kappa_score

In [ ]:
# Files are expected to be in the same folder as this notebook.
INPUT_FILE = Path("manual_review.xlsx")
SHEET_NAME = "All Comparisons"

FEATURES = [
    "Ontvanger",
    "Besluitvormend Orgaan",
    "Rechtshandeling",
    "Rechtsobject",
]

A_COL = "annotator_1_normalized"
B_COL = "annotator_(2, 3)_normalized"
AGREEMENT_COL = "auto_full_agreement"

In [4]:
df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)

def clean_signature(value):
    if pd.isna(value):
        return "<NONE>"

    value = str(value).strip()

    if not value or value.lower() == "nan":
        return "<NONE>"

    return value


df["kappa_A"] = df[A_COL].apply(clean_signature)
df["kappa_B"] = df[B_COL].apply(clean_signature)

df[AGREEMENT_COL] = (
    pd.to_numeric(df[AGREEMENT_COL], errors="coerce")
    .fillna(0)
    .astype(int)
)

agreement_mask = df[AGREEMENT_COL] == 1
df.loc[agreement_mask, "kappa_B"] = df.loc[agreement_mask, "kappa_A"]

In [5]:
results = []

for feature in FEATURES:
    subset = df[df["feature"] == feature]

    y_a = subset["kappa_A"].tolist()
    y_b = subset["kappa_B"].tolist()

    n = len(subset)
    agreements = sum(a == b for a, b in zip(y_a, y_b))
    agreement_pct = agreements / n * 100 if n else 0
    kappa = cohen_kappa_score(y_a, y_b) if n else float("nan")

    results.append({
        "Feature": feature,
        "N": n,
        "Agreements": agreements,
        "Agreement (%)": agreement_pct,
        "Cohen's kappa": kappa,
    })


overall_a = (
    df["feature"].astype(str) + "::" + df["kappa_A"].astype(str)
).tolist()

overall_b = (
    df["feature"].astype(str) + "::" + df["kappa_B"].astype(str)
).tolist()

overall_n = len(df)
overall_agreements = sum(a == b for a, b in zip(overall_a, overall_b))
overall_agreement_pct = (
    overall_agreements / overall_n * 100 if overall_n else 0
)
overall_kappa = (
    cohen_kappa_score(overall_a, overall_b)
    if overall_n
    else float("nan")
)

results.append({
    "Feature": "Overall",
    "N": overall_n,
    "Agreements": overall_agreements,
    "Agreement (%)": overall_agreement_pct,
    "Cohen's kappa": overall_kappa,
})

In [6]:
kappa_results = pd.DataFrame(results)

kappa_results["Agreement (%)"] = kappa_results["Agreement (%)"].round(1)
kappa_results["Cohen's kappa"] = kappa_results["Cohen's kappa"].round(3)

kappa_results

,Feature,N,Agreements,Agreement (%),Cohen's kappa
0,Ontvanger,140,100,71.4,0.710
1,Besluitvormend Orgaan,140,134,95.7,0.953
2,Rechtshandeling,140,131,93.6,0.926
3,Rechtsobject,140,134,95.7,0.940
4,Overall,560,499,89.1,0.887
